In [1]:
import pandas as pd
import numpy as np
import re
import nltk
import random

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.sentiment import SentimentIntensityAnalyzer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA

from qiskit_aer.primitives import Sampler
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.algorithms.classifiers import VQC
from qiskit_algorithms.optimizers import COBYLA

# --------- REPRODUCIBILITY ----------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# --------- NLTK RESOURCES ----------
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('vader_lexicon')

# --------- LOAD DATASET ----------
# Place AutismData.csv in the SAME folder as this notebook
df = pd.read_csv("AutismData.csv")
df.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')

TEXT_COL = "statement"

# --------- AUTISM-SPECIFIC FILTERING ----------
autism_keywords = [
    "autism", "autistic", "asd", "spectrum",
    "sensory", "meltdown", "stimming",
    "communication", "social"
]

def is_autism_related(text):
    text = str(text).lower()
    return any(word in text for word in autism_keywords)

df = df[df[TEXT_COL].apply(is_autism_related)].reset_index(drop=True)

# --------- TEXT CLEANING ----------
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    return " ".join(tokens)

df[TEXT_COL] = df[TEXT_COL].apply(clean_text)

# --------- BINARY SENTIMENT LABELING ----------
sia = SentimentIntensityAnalyzer()

def get_binary_sentiment(text):
    score = sia.polarity_scores(text)['compound']
    return "Positive" if score >= 0 else "Negative"

df["sentiment"] = df[TEXT_COL].apply(get_binary_sentiment)

# --------- ENCODE LABELS ----------
le = LabelEncoder()
y = le.fit_transform(df["sentiment"])

# --------- TF-IDF FEATURES ----------
vectorizer = TfidfVectorizer(
    max_features=300,
    ngram_range=(1, 2)
)
X = vectorizer.fit_transform(df[TEXT_COL]).toarray()

# --------- TRAIN / TEST SPLIT ----------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

# --------- PCA (QUANTUM-COMPATIBLE) ----------
pca = PCA(n_components=4, random_state=SEED)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

# ===================== CLASSICAL MODEL =====================
classical_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

classical_model.fit(X_train_pca, y_train)
y_pred_classical = classical_model.predict(X_test_pca)

print("\n--- Classical NLP Sentiment Model (Binary, Balanced) ---")
print("Accuracy:", accuracy_score(y_test, y_pred_classical))
print(classification_report(y_test, y_pred_classical, target_names=le.classes_))

# ===================== QUANTUM MODEL =====================
feature_map = ZZFeatureMap(
    feature_dimension=4,
    reps=1,
    entanglement="linear"
)

optimizer = COBYLA(maxiter=60)
sampler = Sampler()

quantum_model = VQC(
    feature_map=feature_map,
    optimizer=optimizer,
    sampler=sampler
)

quantum_model.fit(X_train_pca, y_train)
y_pred_quantum = quantum_model.predict(X_test_pca)

print("\n--- Quantum Sentiment Model (VQC, Binary) ---")
print("Accuracy:", accuracy_score(y_test, y_pred_quantum))
print(classification_report(y_test, y_pred_quantum, target_names=le.classes_))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kusha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\kusha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\kusha\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!



--- Classical NLP Sentiment Model (Binary, Balanced) ---
Accuracy: 0.553875236294896
              precision    recall  f1-score   support

    Negative       0.66      0.58      0.62       327
    Positive       0.43      0.51      0.47       202

    accuracy                           0.55       529
   macro avg       0.54      0.55      0.54       529
weighted avg       0.57      0.55      0.56       529


--- Quantum Sentiment Model (VQC, Binary) ---
Accuracy: 0.6200378071833649
              precision    recall  f1-score   support

    Negative       0.62      0.98      0.76       327
    Positive       0.53      0.04      0.07       202

    accuracy                           0.62       529
   macro avg       0.58      0.51      0.42       529
weighted avg       0.59      0.62      0.50       529



In [3]:
# ===================== IMPROVED SENTIMENT ANALYSIS OF AUTISM-AFFECTED CHILDREN
#          USING NLP + QUANTUM MACHINE LEARNING (JUPYTER)
# =====================

import pandas as pd
import numpy as np
import re
import nltk
import random
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.svm import LinearSVC
from sklearn.decomposition import PCA
from qiskit_aer.primitives import Sampler
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.algorithms.classifiers import VQC
from qiskit_algorithms.optimizers import COBYLA

# ------------------ REPRODUCIBILITY ------------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# ------------------ NLTK ------------------
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('vader_lexicon')

# ------------------ LOAD DATA ------------------
df = pd.read_csv("AutismData.csv")
df.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')

TEXT_COL = "statement"

# ------------------ AUTISM FILTER ------------------
autism_keywords = [
    "autism", "autistic", "asd", "spectrum",
    "sensory", "meltdown", "stimming",
    "communication", "social"
]

def is_autism_related(text):
    text = str(text).lower()
    return any(word in text for word in autism_keywords)

df = df[df[TEXT_COL].apply(is_autism_related)].reset_index(drop=True)

# ------------------ TEXT CLEANING ------------------
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    return " ".join(tokens)

df[TEXT_COL] = df[TEXT_COL].apply(clean_text)

# Remove very short / noisy samples
df = df[df[TEXT_COL].str.split().str.len() >= 5]

# ------------------ BINARY SENTIMENT ------------------
sia = SentimentIntensityAnalyzer()

def get_binary_sentiment(text):
    score = sia.polarity_scores(text)['compound']
    return "Positive" if score >= 0 else "Negative"

df["sentiment"] = df[TEXT_COL].apply(get_binary_sentiment)

# ------------------ ENCODE LABELS ------------------
le = LabelEncoder()
y = le.fit_transform(df["sentiment"])

# ------------------ TF-IDF (IMPROVED) ------------------
vectorizer = TfidfVectorizer(
    max_features=600,
    ngram_range=(1, 1),     # unigrams only
    min_df=3,
    sublinear_tf=True
)
X = vectorizer.fit_transform(df[TEXT_COL]).toarray()

# ------------------ TRAIN / TEST SPLIT ------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

# ------------------ PCA (LESS AGGRESSIVE) ------------------
pca = PCA(n_components=8, random_state=SEED)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

# ===================== CLASSICAL MODEL (STRONGER) =====================
classical_model = LinearSVC(class_weight="balanced")

classical_model.fit(X_train_pca, y_train)
y_pred_classical = classical_model.predict(X_test_pca)

print("\n--- Improved Classical NLP Sentiment Model ---")
print("Accuracy:", accuracy_score(y_test, y_pred_classical))
print(classification_report(y_test, y_pred_classical, target_names=le.classes_))

# ===================== QUANTUM MODEL (FAIR COMPARISON) =====================
feature_map = ZZFeatureMap(
    feature_dimension=8,
    reps=1,
    entanglement="linear"
)

optimizer = COBYLA(maxiter=80)
sampler = Sampler()

quantum_model = VQC(
    feature_map=feature_map,
    optimizer=optimizer,
    sampler=sampler
)

quantum_model.fit(X_train_pca, y_train)
y_pred_quantum = quantum_model.predict(X_test_pca)

print("\n--- Quantum Sentiment Model (VQC, Improved) ---")
print("Accuracy:", accuracy_score(y_test, y_pred_quantum))
print(classification_report(y_test, y_pred_quantum, target_names=le.classes_))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kusha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\kusha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\kusha\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!



--- Improved Classical NLP Sentiment Model ---
Accuracy: 0.6559546313799622
              precision    recall  f1-score   support

    Negative       0.76      0.65      0.70       327
    Positive       0.54      0.67      0.60       202

    accuracy                           0.66       529
   macro avg       0.65      0.66      0.65       529
weighted avg       0.68      0.66      0.66       529


--- Quantum Sentiment Model (VQC, Improved) ---
Accuracy: 0.6238185255198487
              precision    recall  f1-score   support

    Negative       0.67      0.79      0.72       327
    Positive       0.51      0.36      0.42       202

    accuracy                           0.62       529
   macro avg       0.59      0.57      0.57       529
weighted avg       0.61      0.62      0.61       529

